In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load both datasets
wildfire = pd.read_csv('../data/wildfire_weather.csv', low_memory=False)
insurance = pd.read_csv('../data/insurance_fire_census_weather_raw.csv', low_memory=False)

# Load feature descriptions
feat_wildfire = pd.read_csv('../data/Feature_Descsription_FireHistory_Census.csv', encoding='latin1', low_memory=False)
feat_insurance = pd.read_csv('../data/FeatureDescription_fire_insurance.csv', encoding='latin1', low_memory=False)

print("=== Wildfire Dataset ===")
print(f"Shape: {wildfire.shape}")
print(f"Years: {sorted(wildfire['Year'].dropna().unique())}")
print(f"Unique zip codes: {wildfire['zip'].nunique()}")
print(f"\nRows with fire events: {wildfire['FIRE_NAME'].notna().sum()}")
print(f"Rows without fire (weather only): {wildfire['FIRE_NAME'].isna().sum()}")

print("\n\n=== Insurance Dataset ===")
print(f"Shape: {insurance.shape}")
print(f"Years: {sorted(insurance['Year'].dropna().unique())}")
print(f"Unique zip codes: {insurance['ZIP'].nunique()}")
print(f"Target column (Earned Premium) - non-null: {insurance['Earned Premium'].notna().sum()}")

=== Wildfire Dataset ===
Shape: (125476, 30)
Years: [np.float64(2018.0), np.float64(2019.0), np.float64(2020.0), np.float64(2021.0), np.float64(2022.0), np.float64(2023.0)]
Unique zip codes: 2599

Rows with fire events: 2211
Rows without fire (weather only): 123265


=== Insurance Dataset ===
Shape: (47033, 76)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
Unique zip codes: 2251
Target column (Earned Premium) - non-null: 47033


In [2]:
print("=== Wildfire Columns ===")
for col in wildfire.columns:
    non_null = wildfire[col].notna().sum()
    dtype = wildfire[col].dtype
    sample = wildfire[col].dropna().iloc[0] if non_null > 0 else "ALL NULL"
    print(f"  {col:25s} | {str(dtype):10s} | {non_null:>6d} non-null | example: {sample}")

=== Wildfire Columns ===
  OBJECTID                  | float64    |   2218 non-null | example: 1653.0
  Year                      | float64    |   2218 non-null | example: 2019.0
  AGENCY                    | str        |   2218 non-null | example: CDF
  UNIT_ID                   | str        |   2209 non-null | example: MEU
  FIRE_NAME                 | str        |   2211 non-null | example: GRADE
  INC_NUM                   | str        |   2186 non-null | example: 00008307
  ALARM_DATE                | str        |   2212 non-null | example: 7/16/17
  CONT_DATE                 | str        |   2193 non-null | example: 7/21/17
  CAUSE                     | float64    |   2218 non-null | example: 10.0
  C_METHOD                  | float64    |   2216 non-null | example: 7.0
  OBJECTIVE                 | float64    |   2212 non-null | example: 1.0
  GIS_ACRES                 | float64    |   2218 non-null | example: 890.5153
  COMMENTS                  | str        |    460 non-null |

In [3]:
print("=== Insurance Columns ===")
for col in insurance.columns:
    non_null = insurance[col].notna().sum()
    dtype = insurance[col].dtype
    sample = insurance[col].dropna().iloc[0] if non_null > 0 else "ALL NULL"
    print(f"  {col:45s} | {str(dtype):10s} | {non_null:>6d} non-null | example: {sample}")

=== Insurance Columns ===
  Year                                          | int64      |  47033 non-null | example: 2018
  ZIP                                           | int64      |  47033 non-null | example: 90003
  Avg Fire Risk Score                           | float64    |  47033 non-null | example: 1.0
  Avg PPC                                       | float64    |  19889 non-null | example: 1.0
  CAT Cov A Fire -  Incurred Losses             | int64      |  47033 non-null | example: 0
  CAT Cov A Fire -  Number of Claims            | int64      |  47033 non-null | example: 0
  CAT Cov A Smoke -  Incurred Losses            | int64      |  47033 non-null | example: 0
  CAT Cov A Smoke -  Number of Claims           | int64      |  47033 non-null | example: 0
  CAT Cov C Fire -  Incurred Losses             | int64      |  47033 non-null | example: 0
  CAT Cov C Fire -  Number of Claims            | int64      |  47033 non-null | example: 0
  CAT Cov C Smoke -  Incurred Losses       

In [4]:
# === WILDFIRE DATASET: Understanding the structure ===

# Separate fire events from weather-only rows
fire_events = wildfire[wildfire['FIRE_NAME'].notna()].copy()
weather_only = wildfire[wildfire['FIRE_NAME'].isna()].copy()

print("=== Fire Events Breakdown ===")
print(f"Total fire incidents: {len(fire_events)}")
print(f"\nFires per year:")
print(fire_events['Year'].value_counts().sort_index())
print(f"\nZip codes that had at least one fire: {fire_events['zip'].nunique()}")
print(f"Zip codes that NEVER had a fire: {wildfire['zip'].nunique() - fire_events['zip'].nunique()}")

# How many zip codes have fires in each year?
fire_zips_per_year = fire_events.groupby('Year')['zip'].nunique()
print(f"\nUnique zip codes with fires per year:")
print(fire_zips_per_year)

# Average fire size
print(f"\n=== Fire Size (GIS_ACRES) ===")
print(fire_events['GIS_ACRES'].describe())

=== Fire Events Breakdown ===
Total fire incidents: 2211

Fires per year:
Year
2018.0    412
2019.0    319
2020.0    502
2021.0    388
2022.0    306
2023.0    284
Name: count, dtype: int64

Zip codes that had at least one fire: 559
Zip codes that NEVER had a fire: 2040

Unique zip codes with fires per year:
Year
2018.0    224
2019.0    178
2020.0    254
2021.0    224
2022.0    173
2023.0    119
Name: zip, dtype: int64

=== Fire Size (GIS_ACRES) ===
count    2.211000e+03
mean     4.171012e+03
std      3.774184e+04
min      1.514838e-03
25%      8.922028e+00
50%      2.864047e+01
75%      1.535234e+02
max      1.032700e+06
Name: GIS_ACRES, dtype: float64


In [7]:
# === FIXED: Build zip-year level view using ALL zip codes ===

# Extract year from year_month for weather rows (e.g., "2018-01" -> 2018)
wildfire['derived_year'] = wildfire['year_month'].str[:4].astype(float)

# Get all unique zip-year combinations from the FULL dataset
all_zips = wildfire[['zip', 'derived_year']].dropna().drop_duplicates()
all_zips = all_zips.rename(columns={'derived_year': 'Year'})

# Mark which zip-years had a fire
fire_events = wildfire[wildfire['FIRE_NAME'].notna()].copy()
fire_zips_years = fire_events[['zip', 'Year']].dropna().drop_duplicates()
fire_zips_years['had_fire'] = 1

# Merge
zip_year = all_zips.merge(fire_zips_years, on=['zip', 'Year'], how='left')
zip_year['had_fire'] = zip_year['had_fire'].fillna(0).astype(int)

print("=== CORRECTED Class Balance (zip-year level) ===")
print(f"Total zip-year combinations: {len(zip_year)}")
print(f"\nFire vs No Fire:")
print(zip_year['had_fire'].value_counts().sort_index())
print(f"\nFire rate: {zip_year['had_fire'].mean():.2%}")

print(f"\n=== By Year ===")
for year in sorted(zip_year['Year'].unique()):
    subset = zip_year[zip_year['Year'] == year]
    total = len(subset)
    fires = subset['had_fire'].sum()
    print(f"  {int(year)}: {fires:>4d} fires / {total:>5d} zips = {fires/total:.2%}")

=== CORRECTED Class Balance (zip-year level) ===
Total zip-year combinations: 10675

Fire vs No Fire:
had_fire
0    9503
1    1172
Name: count, dtype: int64

Fire rate: 10.98%

=== By Year ===
  2017:    0 fires /     3 zips = 0.00%
  2018:  224 fires /  2593 zips = 8.64%
  2019:  178 fires /  2596 zips = 6.86%
  2020:  254 fires /  2597 zips = 9.78%
  2021:  224 fires /  2594 zips = 8.64%
  2022:  173 fires /   173 zips = 100.00%
  2023:  119 fires /   119 zips = 100.00%


In [6]:
# === Weather Features ===
print("=== Weather Data Coverage ===")
weather_monthly = wildfire[['zip', 'year_month', 'avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']].copy()
weather_monthly = weather_monthly[weather_monthly['avg_tmax_c'].notna()]
print(f"Monthly weather records: {len(weather_monthly)}")
print(f"Zip codes with weather data: {weather_monthly['zip'].nunique()}")
print(f"Date range: {weather_monthly['year_month'].min()} to {weather_monthly['year_month'].max()}")

print("\n=== Weather Summary Stats ===")
print(weather_monthly[['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']].describe().round(2))

=== Weather Data Coverage ===
Monthly weather records: 124604
Zip codes with weather data: 2593
Date range: 2018-01 to 2021-12

=== Weather Summary Stats ===
       avg_tmax_c  avg_tmin_c  tot_prcp_mm
count   124604.00   124604.00    124604.00
mean        23.29       10.94        25.04
std          6.97        5.67        42.08
min        -14.92      -23.58         0.00
25%         18.83        7.11         0.00
50%         22.68       11.09         1.80
75%         27.39       14.86        36.50
max         44.71       32.15       366.70


In [8]:
# === Verify: do 2022 and 2023 have weather data for non-fire zip codes? ===
for year in [2018, 2019, 2020, 2021, 2022, 2023]:
    year_data = wildfire[wildfire['derived_year'] == year]
    has_weather = year_data['avg_tmax_c'].notna().sum()
    has_fire = year_data['FIRE_NAME'].notna().sum()
    total_zips = year_data['zip'].nunique()
    print(f"{year}: {total_zips:>5d} zips | {has_weather:>6d} weather rows | {has_fire:>4d} fire rows")

2018:  2593 zips |  31164 weather rows |  409 fire rows
2019:  2596 zips |  31153 weather rows |  313 fire rows
2020:  2597 zips |  31134 weather rows |  501 fire rows
2021:  2594 zips |  31153 weather rows |  389 fire rows
2022:   173 zips |      0 weather rows |  306 fire rows
2023:   119 zips |      0 weather rows |  284 fire rows


In [9]:
# === Weather patterns for zip codes WITH vs WITHOUT fires (2018-2021) ===

# Aggregate weather to zip-year level for 2018-2021
weather_rows = wildfire[
    (wildfire['derived_year'].isin([2018, 2019, 2020, 2021])) & 
    (wildfire['avg_tmax_c'].notna())
].copy()

weather_zip_year = weather_rows.groupby(['zip', 'derived_year']).agg(
    mean_tmax=('avg_tmax_c', 'mean'),
    max_tmax=('avg_tmax_c', 'max'),
    mean_tmin=('avg_tmin_c', 'mean'),
    total_precip=('tot_prcp_mm', 'sum'),
    min_precip=('tot_prcp_mm', 'min'),
).reset_index().rename(columns={'derived_year': 'Year'})

# Join with fire labels
weather_zip_year = weather_zip_year.merge(
    zip_year[zip_year['Year'].isin([2018, 2019, 2020, 2021])][['zip', 'Year', 'had_fire']],
    on=['zip', 'Year'],
    how='left'
)
weather_zip_year['had_fire'] = weather_zip_year['had_fire'].fillna(0).astype(int)

# Compare
print("=== Weather: Fire vs No-Fire zip codes (2018-2021 avg) ===\n")
comparison = weather_zip_year.groupby('had_fire')[
    ['mean_tmax', 'max_tmax', 'mean_tmin', 'total_precip', 'min_precip']
].mean().round(2)
comparison.index = ['No Fire', 'Had Fire']
print(comparison.to_string())

=== Weather: Fire vs No-Fire zip codes (2018-2021 avg) ===

          mean_tmax  max_tmax  mean_tmin  total_precip  min_precip
No Fire       23.27     31.52      11.03        302.55        0.36
Had Fire      23.54     33.22      10.01        281.61        0.62


In [10]:
# === Can fire HISTORY predict future fires? ===
# If a zip had a fire in prior years, is it more likely to have one again?

fire_events_clean = wildfire[
    (wildfire['FIRE_NAME'].notna()) & 
    (wildfire['Year'].isin([2018, 2019, 2020, 2021]))
].copy()

# Count fires per zip across all years
fire_counts = fire_events_clean.groupby('zip').agg(
    total_fires=('FIRE_NAME', 'count'),
    total_acres_burned=('GIS_ACRES', 'sum'),
    max_fire_size=('GIS_ACRES', 'max'),
    avg_fire_size=('GIS_ACRES', 'mean')
).reset_index()

print("=== Fire History Distribution (2018-2021) ===")
print(f"Zip codes with at least one fire: {len(fire_counts)}")
print(f"\nFires per zip code:")
print(fire_counts['total_fires'].describe().round(2))

# Do repeat-fire zip codes exist?
print(f"\nZip codes with 1 fire: {(fire_counts['total_fires'] == 1).sum()}")
print(f"Zip codes with 2+ fires: {(fire_counts['total_fires'] >= 2).sum()}")
print(f"Zip codes with 5+ fires: {(fire_counts['total_fires'] >= 5).sum()}")
print(f"Zip codes with 10+ fires: {(fire_counts['total_fires'] >= 10).sum()}")

# Does having a fire in 2018-2020 predict having one in 2021?
fire_1820 = wildfire[
    (wildfire['FIRE_NAME'].notna()) & (wildfire['Year'].isin([2018, 2019, 2020]))
]['zip'].unique()
fire_2021 = wildfire[
    (wildfire['FIRE_NAME'].notna()) & (wildfire['Year'] == 2021)
]['zip'].unique()

had_prior = set(fire_1820)
had_2021 = set(fire_2021)
all_zips_set = set(wildfire[wildfire['derived_year'].isin([2018,2019,2020,2021])]['zip'].dropna().unique())

tp = len(had_prior & had_2021)  # had prior fire AND had 2021 fire
fn = len(had_2021 - had_prior)  # had 2021 fire but NO prior fire
fp = len(had_prior - had_2021)  # had prior fire but NO 2021 fire
tn = len(all_zips_set - had_prior - had_2021)  # no prior, no 2021

print(f"\n=== Does prior fire (2018-2020) predict 2021 fire? ===")
print(f"Had prior fire AND had 2021 fire:  {tp}")
print(f"Had prior fire, NO 2021 fire:      {fp}")
print(f"NO prior fire, had 2021 fire:       {fn}")
print(f"NO prior fire, NO 2021 fire:        {tn}")
print(f"\nPrecision (prior fire -> 2021 fire): {tp/(tp+fp):.2%}")
print(f"Recall (2021 fires caught by prior): {tp/(tp+fn):.2%}")

=== Fire History Distribution (2018-2021) ===
Zip codes with at least one fire: 495

Fires per zip code:
count    495.00
mean       2.83
std        2.96
min        1.00
25%        1.00
50%        2.00
75%        3.00
max       28.00
Name: total_fires, dtype: float64

Zip codes with 1 fire: 207
Zip codes with 2+ fires: 288
Zip codes with 5+ fires: 75
Zip codes with 10+ fires: 19

=== Does prior fire (2018-2020) predict 2021 fire? ===
Had prior fire AND had 2021 fire:  168
Had prior fire, NO 2021 fire:      272
NO prior fire, had 2021 fire:       57
NO prior fire, NO 2021 fire:        2104

Precision (prior fire -> 2021 fire): 38.18%
Recall (2021 fires caught by prior): 74.67%


In [11]:
# === Insurance dataset: useful features for fire prediction? ===

ins_2020 = insurance[insurance['Year'] == 2020].copy()

print("=== Insurance Features (2020 snapshot) ===")
print(f"Zip codes: {ins_2020['ZIP'].nunique()}")
print(f"\nAvg Fire Risk Score:")
print(ins_2020['Avg Fire Risk Score'].describe().round(3))

print(f"\n=== Fire Risk Score vs Actual Fires ===")
# Match insurance zip codes to actual fire occurrence
fire_2021_zips = set(wildfire[(wildfire['FIRE_NAME'].notna()) & (wildfire['Year'] == 2021)]['zip'].dropna().astype(int))

ins_2020['actual_fire_2021'] = ins_2020['ZIP'].isin(fire_2021_zips).astype(int)

comparison = ins_2020.groupby('actual_fire_2021').agg(
    avg_fire_risk=('Avg Fire Risk Score', 'mean'),
    avg_earned_premium=('Earned Premium', 'mean'),
    avg_housing_value=('housing_value', 'mean'),
    avg_population=('total_population', 'mean'),
    avg_median_income=('median_income', 'mean'),
    count=('ZIP', 'count')
).round(2)
comparison.index = ['No Fire 2021', 'Had Fire 2021']
print(comparison.to_string())

=== Insurance Features (2020 snapshot) ===
Zip codes: 2118

Avg Fire Risk Score:
count    13800.000
mean         0.788
std          0.860
min          0.000
25%          0.010
50%          0.510
75%          1.060
max          4.000
Name: Avg Fire Risk Score, dtype: float64

=== Fire Risk Score vs Actual Fires ===
               avg_fire_risk  avg_earned_premium  avg_housing_value  avg_population  avg_median_income  count
No Fire 2021            0.71           491532.80          752232.95        22284.77           98633.39  11682
Had Fire 2021           1.22           720387.42          580422.30        19414.36           88994.43   2118


In [12]:
# === Fire causes and geographic patterns ===

print("=== Fire Causes (2018-2021) ===")
cause_map = {
    1: 'Lightning', 2: 'Equipment Use', 3: 'Smoking', 4: 'Campfire',
    5: 'Debris', 6: 'Railroad', 7: 'Arson', 8: 'Playing with fire',
    9: 'Miscellaneous', 10: 'Vehicle', 11: 'Power Line', 12: 'Firefighter Training',
    13: 'Non-Firefighter Training', 14: 'Unknown/Unidentified',
    15: 'Escaped Prescribed Burn', 16: 'Illegal Alien Campfire',
    17: 'Escaped Control Burn', 18: 'Structure', 19: 'Aircraft'
}
fire_events_clean['cause_name'] = fire_events_clean['CAUSE'].map(cause_map)
print(fire_events_clean['cause_name'].value_counts().head(10))

print(f"\n=== Geographic spread ===")
print(f"Latitude range: {fire_events_clean['latitude'].min():.2f} to {fire_events_clean['latitude'].max():.2f}")
print(f"Longitude range: {fire_events_clean['longitude'].min():.2f} to {fire_events_clean['longitude'].max():.2f}")

# Which agencies respond most?
print(f"\n=== Responding Agencies ===")
print(fire_events_clean['AGENCY'].value_counts())

=== Fire Causes (2018-2021) ===
cause_name
Unknown/Unidentified    552
Lightning               232
Equipment Use           207
Miscellaneous           164
Vehicle                 126
Power Line              107
Arson                    78
Debris                   75
Campfire                 27
Playing with fire        21
Name: count, dtype: int64

=== Geographic spread ===
Latitude range: 32.56 to 42.01
Longitude range: -124.22 to -114.14

=== Responding Agencies ===
AGENCY
CDF    815
CCO    284
USF    283
BLM     84
LRA     67
NPS     45
FWS     25
DOD      9
BIA      9
Name: count, dtype: int64


In [13]:
# === Insurance: Fire Exposure Risk Categories ===
ins_2020 = insurance[insurance['Year'] == 2020].copy()
fire_2021_zips = set(wildfire[(wildfire['FIRE_NAME'].notna()) & (wildfire['Year'] == 2021)]['zip'].dropna().astype(int))
ins_2020['actual_fire_2021'] = ins_2020['ZIP'].isin(fire_2021_zips).astype(int)

exposure_cols = [
    'Number of High Fire Risk Exposure',
    'Number of Very High Fire Risk Exposure', 
    'Number of Low Fire Risk Exposure',
    'Number of Moderate Fire Risk Exposure',
    'Number of Negligible Fire Risk Exposure'
]
print("=== Fire Risk Exposure Categories (2020) ===")
comp = ins_2020.groupby('actual_fire_2021')[exposure_cols].mean().round(2)
comp.index = ['No Fire 2021', 'Had Fire 2021']
print(comp.to_string())

=== Fire Risk Exposure Categories (2020) ===
               Number of High Fire Risk Exposure  Number of Very High Fire Risk Exposure  Number of Low Fire Risk Exposure  Number of Moderate Fire Risk Exposure  Number of Negligible Fire Risk Exposure
No Fire 2021                               24.36                                    2.53                            209.19                                  30.10                                   293.63
Had Fire 2021                              64.61                                    7.20                            288.12                                  77.72                                   265.23


In [14]:
# === Insurance: PPC (Public Protection Classification) ===
# Lower PPC = better fire protection (1 is best, 10 is worst)
print("=== Avg PPC (fire department quality) ===")
print(ins_2020.groupby('actual_fire_2021')['Avg PPC'].mean().round(3))
print(f"\nPPC distribution:")
print(ins_2020['Avg PPC'].describe().round(2))
print(f"Non-null: {ins_2020['Avg PPC'].notna().sum()} / {len(ins_2020)}")

=== Avg PPC (fire department quality) ===
actual_fire_2021
0   NaN
1   NaN
Name: Avg PPC, dtype: float64

PPC distribution:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: Avg PPC, dtype: float64
Non-null: 0 / 13800


In [15]:
# === Insurance: Claims patterns ===
claims_cols = [
    'CAT Cov A Fire -  Incurred Losses', 'CAT Cov A Fire -  Number of Claims',
    'Non-CAT Cov A Fire -  Incurred Losses', 'Non-CAT Cov A Fire -  Number of Claims',
    'CAT Cov A Smoke -  Incurred Losses', 'CAT Cov A Smoke -  Number of Claims',
]
print("=== Claims: Fire vs No-Fire zips (2020) ===")
comp = ins_2020.groupby('actual_fire_2021')[claims_cols].mean().round(2)
comp.index = ['No Fire 2021', 'Had Fire 2021']
print(comp.to_string())

# What % of zip codes have ANY claims?
print(f"\n=== Zip codes with any CAT fire claims (2020) ===")
has_cat = (ins_2020['CAT Cov A Fire -  Number of Claims'] > 0).sum()
print(f"{has_cat} / {len(ins_2020)} rows ({has_cat/len(ins_2020):.2%})")

=== Claims: Fire vs No-Fire zips (2020) ===
               CAT Cov A Fire -  Incurred Losses  CAT Cov A Fire -  Number of Claims  Non-CAT Cov A Fire -  Incurred Losses  Non-CAT Cov A Fire -  Number of Claims  CAT Cov A Smoke -  Incurred Losses  CAT Cov A Smoke -  Number of Claims
No Fire 2021                            69421.96                                0.47                               51674.36                                    0.51                             6299.15                                 0.58
Had Fire 2021                          692673.78                                2.05                               87221.03                                    0.85                            16948.30                                 0.99

=== Zip codes with any CAT fire claims (2020) ===
1018 / 13800 rows (7.38%)


In [16]:
# === Insurance: Category columns ===
cat_cols = ['Category_CO', 'Category_DO', 'Category_DT', 'Category_HO', 'Category_MH', 'Category_RT']
print("=== Insurance Categories (what are these?) ===")
for col in cat_cols:
    true_count = ins_2020[col].sum()
    print(f"  {col}: {true_count} True ({true_count/len(ins_2020):.1%})")

print(f"\n=== Category breakdown for fire vs no-fire ===")
comp = ins_2020.groupby('actual_fire_2021')[cat_cols].mean().round(3)
comp.index = ['No Fire 2021', 'Had Fire 2021']
print(comp.to_string())

=== Insurance Categories (what are these?) ===
  Category_CO: 2300 True (16.7%)
  Category_DO: 2300 True (16.7%)
  Category_DT: 2300 True (16.7%)
  Category_HO: 2300 True (16.7%)
  Category_MH: 2300 True (16.7%)
  Category_RT: 2300 True (16.7%)

=== Category breakdown for fire vs no-fire ===
               Category_CO  Category_DO  Category_DT  Category_HO  Category_MH  Category_RT
No Fire 2021         0.167        0.167        0.167        0.167        0.167        0.167
Had Fire 2021        0.167        0.167        0.167        0.167        0.167        0.167


In [17]:
# === Wildfire: Seasonality and fire size patterns ===
fire_events_clean = wildfire[
    (wildfire['FIRE_NAME'].notna()) & 
    (wildfire['Year'].isin([2018, 2019, 2020, 2021]))
].copy()
fire_events_clean['alarm_month'] = pd.to_datetime(fire_events_clean['Alarm_Date2']).dt.month

print("=== Fire Seasonality (month of alarm) ===")
print(fire_events_clean['alarm_month'].value_counts().sort_index())

print(f"\n=== Total Acres Burned per Zip (top 20 worst zips) ===")
acres_by_zip = fire_events_clean.groupby('zip')['GIS_ACRES'].agg(['sum', 'count', 'max']).round(1)
acres_by_zip.columns = ['total_acres', 'fire_count', 'largest_fire']
acres_by_zip = acres_by_zip.sort_values('total_acres', ascending=False)
print(acres_by_zip.head(20).to_string())

print(f"\n=== Agency distribution for fire vs repeat-fire zips ===")
repeat_zips = set(acres_by_zip[acres_by_zip['fire_count'] >= 3].index)
fire_events_clean['is_repeat_zip'] = fire_events_clean['zip'].isin(repeat_zips)
print(fire_events_clean.groupby('is_repeat_zip')['AGENCY'].value_counts().unstack(fill_value=0))

=== Fire Seasonality (month of alarm) ===
alarm_month
1.0      41
2.0      29
3.0      24
4.0      56
5.0     126
6.0     344
7.0     338
8.0     283
9.0     183
10.0    119
11.0     51
12.0     21
Name: count, dtype: int64

=== Total Acres Burned per Zip (top 20 worst zips) ===
         total_acres  fire_count  largest_fire
zip                                           
96137.0     963405.4           1      963405.4
94550.0     396894.5           2      396824.5
93634.0     380549.0           4      379842.4
94558.0     308215.0          13      305351.9
96033.0     229651.4           1      229651.4
96010.0     223134.6           3      223107.9
96031.0     199579.7           2      199354.1
93265.0     185582.0           7      170647.9
95965.0     154576.8           8      153335.6
96105.0     151992.3           7      105003.9
95527.0     143836.4           1      143836.4
95448.0     132993.5           9       77762.1
93928.0     124527.4           1      124527.4
96076.0     123

In [18]:
# === Quick check: does PPC have data in any year? ===
for yr in [2018, 2019, 2020, 2021]:
    subset = insurance[insurance['Year'] == yr]
    non_null = subset['Avg PPC'].notna().sum()
    print(f"  {yr}: Avg PPC non-null = {non_null} / {len(subset)}")

# Also check Earned Exposure and Cov A/C weighted averages
print(f"\n=== Other potentially useful columns ===")
for col in ['Earned Exposure', 'Cov A Amount Weighted Avg', 'Cov C Amount Weighted Avg']:
    comp = insurance[insurance['Year'] == 2020].copy()
    fire_2021_zips = set(wildfire[(wildfire['FIRE_NAME'].notna()) & (wildfire['Year'] == 2021)]['zip'].dropna().astype(int))
    comp['fire'] = comp['ZIP'].isin(fire_2021_zips).astype(int)
    means = comp.groupby('fire')[col].mean().round(2)
    print(f"  {col}: No fire={means.get(0, 'N/A')}, Fire={means.get(1, 'N/A')}")

  2018: Avg PPC non-null = 10071 / 10071
  2019: Avg PPC non-null = 9818 / 9818
  2020: Avg PPC non-null = 0 / 13800
  2021: Avg PPC non-null = 0 / 13344

=== Other potentially useful columns ===
  Earned Exposure: No fire=559.81, Fire=702.9
  Cov A Amount Weighted Avg: No fire=196149.41, Fire=219661.68
  Cov C Amount Weighted Avg: No fire=75633.61, Fire=87212.76
